In [1]:
import numpy as np

# Input floating-point array
fp32_array = np.array([[191.6, -13.5, 728.6], [92.14, 295.5, -184], [0, 648.6, 245.5]])

#### Linear quantization - FP32 into INT8
- Scaling Factor
```S = (Wmax - Wmin) / (Qmax - Qmin)``` 

In [2]:
S = (fp32_array.max() - fp32_array.min()) / (127 - (-128))
S

3.578823529411765

- Zero point
```Z = Qmin - round(Wmin/S)```

In [3]:
Z = round(-128 - (fp32_array.min() / S))
Z

-77

- Quantize
```Q = round(s * x + z)```

In [4]:
Q = np.round((fp32_array / S) + Z)
Q

array([[ -23.,  -81.,  127.],
       [ -51.,    6., -128.],
       [ -77.,  104.,   -8.]])

- De quantize
``` W = S * (Q - Z)```

In [5]:
S * (Q - Z)

array([[ 193.25647059,  -14.31529412,  730.08      ],
       [  93.04941176,  297.04235294, -182.52      ],
       [   0.        ,  647.76705882,  246.93882353]])

- Quantization Error

In [6]:
np.round(np.abs(fp32_array - (S * (Q - Z))), 2)

array([[1.66, 0.82, 1.48],
       [0.91, 1.54, 1.48],
       [0.  , 0.83, 1.44]])

#### Using Quanto

In [7]:
# input_text = "Hello, my name is "
# input_ids = tokenizer(input_text, return_tensors="pt").input_ids

# outputs = model.generate(input_ids)
# print(tokenizer.decode(outputs[0]))

In [8]:
# from quanto import quantize, freeze
# import torch

# quantize(model, weights=torch.int8, activations=None)
# print(model)

# freeze(model)

#### Using Torch

In [9]:
import torch

test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6], [92.14, 295.5, -184], [0, 684.6, 245.5]]
)

- Get Scaling factor and Zero point

In [10]:
dtype = torch.int8
q_min, q_max = torch.iinfo(dtype).min, torch.iinfo(dtype).max
r_min, r_max = test_tensor.min().item(), test_tensor.max().item()

S = (r_max - r_min) / (q_max - q_min)
S

3.578823433670343

In [11]:
Z = q_min - (r_min / S)
if Z < q_min:
    Z = q_min
elif Z > q_max:
    Z = q_max
else:
    # round and cast to int
    Z = int(round(Z))
Z

-77

- Quantize

In [12]:
Q = torch.round((test_tensor / S) + Z).to(dtype)
Q

tensor([[ -23,  -81,  127],
        [ -51,    6, -128],
        [ -77,  114,   -8]], dtype=torch.int8)

- De-quantize

In [13]:
test_tensor_hat = S * (Q - Z)
print(test_tensor)
print(test_tensor_hat)

tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])
tensor([[ 193.2565,  -14.3153, -186.0988],
        [  93.0494,  297.0423, -182.5200],
        [   0.0000, -232.6235,  246.9388]])


- Quantization Error

In [14]:
(test_tensor - test_tensor_hat).abs()

tensor([[1.6564e+00, 8.1529e-01, 9.1470e+02],
        [9.0941e-01, 1.5423e+00, 1.4800e+00],
        [0.0000e+00, 9.1722e+02, 1.4388e+00]])

In [15]:
(test_tensor_hat - test_tensor).abs().square().mean()

tensor(186442.6406)

#### Symmetric vs Assymetric

In [16]:
def quantize(tensor, dtype=torch.int8, symmetric=False):
    r_min, r_max = tensor.min().item(), tensor.max().item()
    q_min, q_max = torch.iinfo(dtype).min, torch.iinfo(dtype).max
    if symmetric:
        q_min = -q_max
        r_min = -r_max

    S = (r_max - r_min) / (q_max - q_min)
    Z = int(round(q_min - (r_min / S)))
    print(f"Scale = {S}, Zero point = {Z}")
    return ((tensor / S) + Z).round()


def dequantize(tensor, S, Z):
    return S * (tensor - Z)


test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6], [92.14, 295.5, -184], [0, 684.6, 245.5]]
)

tensor_q_asym = quantize(tensor=test_tensor, dtype=torch.int8, symmetric=False)
tensor_q_sym = quantize(tensor=test_tensor, dtype=torch.int8, symmetric=True)

tensor_dq_asym = dequantize(tensor_q_asym, S=3.578823433670343, Z=-77)
tensor_dq_sym = dequantize(tensor_q_sym, S=5.737007681779035, Z=0)

print("== Original tensor ==")
print(test_tensor)
print()
print("== Asymmetric Quantized tensor ==")
print(tensor_q_asym)
print()
print("== Asymmetric De-Quantized tensor ==")
print(tensor_dq_asym)
print()
print("== Symmetric Quantized tensor ==")
print(tensor_q_sym)
print()
print("== Symmetric De-Quantized tensor ==")
print(tensor_dq_sym)

Scale = 3.578823433670343, Zero point = -77
Scale = 5.737007681779035, Zero point = 0
== Original tensor ==
tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])

== Asymmetric Quantized tensor ==
tensor([[ -23.,  -81.,  127.],
        [ -51.,    6., -128.],
        [ -77.,  114.,   -8.]])

== Asymmetric De-Quantized tensor ==
tensor([[ 193.2565,  -14.3153,  730.0800],
        [  93.0494,  297.0423, -182.5200],
        [   0.0000,  683.5552,  246.9388]])

== Symmetric Quantized tensor ==
tensor([[ 33.,  -2., 127.],
        [ 16.,  52., -32.],
        [  0., 119.,  43.]])

== Symmetric De-Quantized tensor ==
tensor([[ 189.3213,  -11.4740,  728.6000],
        [  91.7921,  298.3244, -183.5842],
        [   0.0000,  682.7039,  246.6913]])


- Quantization error

In [17]:
def quantization_error(tensor, dequantized_tensor):
    return (dequantized_tensor - tensor).abs().square().mean()

In [18]:
quantization_error(test_tensor, tensor_dq_asym)

tensor(1.5730)

In [19]:
quantization_error(test_tensor, tensor_dq_sym)

tensor(2.5092)

#### Granularities
- Per tensor

In [21]:
test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6], [92.14, 295.5, -184], [0, 684.6, 245.5]]
)

tensor_q_sym = quantize(tensor=test_tensor, dtype=torch.int8, symmetric=True)
tensor_dq_sym = dequantize(tensor_q_sym, S=5.737007681779035, Z=0)
quantization_error(test_tensor, tensor_dq_sym)

Scale = 5.737007681779035, Zero point = 0


tensor(2.5092)

- Per channel

In [34]:
test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6], [92.14, 295.5, -184], [0, 684.6, 245.5]]
)
dim = 0  # 0 - rows, 1 - columns
output_dim = test_tensor.shape[dim]
print(f"# Dim: {output_dim}")

# Initialize scale values
scale = torch.zeros(output_dim)

# Fill the scale values
for index in range(output_dim):
    sub_tensor = test_tensor[index]
    scale[index] = sub_tensor.max().item() / torch.iinfo(torch.int8).max

scale = scale.view((-1, 1))
print("== Scale ==")
print(scale)

print("== Original tensor ==")
print(test_tensor)
print("== Quantized per channel (channel = row) ==")
q_v = ((test_tensor / scale)).round()
print(q_v)
print("== De-Quantized per channel (channel = row) ==")
d_qv = q_v * scale
print(d_qv)

# Dim: 3
== Scale ==
tensor([[5.7370],
        [2.3268],
        [5.3906]])
== Original tensor ==
tensor([[ 191.6000,  -13.5000,  728.6000],
        [  92.1400,  295.5000, -184.0000],
        [   0.0000,  684.6000,  245.5000]])
== Quantized per channel (channel = row) ==
tensor([[ 33.,  -2., 127.],
        [ 40., 127., -79.],
        [  0., 127.,  46.]])
== De-Quantized per channel (channel = row) ==
tensor([[ 189.3213,  -11.4740,  728.6000],
        [  93.0709,  295.5000, -183.8150],
        [   0.0000,  684.6000,  247.9653]])


- Per group

In [40]:
test_tensor = torch.tensor(
    [
        [0.7569, 0.5203, 0.4888, 0.3090],
        [0.6120, 0.2026, 0.3202, 0.9086],
        [0.1879, 0.5149, 0.0355, 0.8984],
        [0.0924, 0.3639, 0.9540, 0.0336],
    ],
    dtype=torch.float32,
)
print(test_tensor)
t_shape = test_tensor.shape
group_size = 2
test_tensor = test_tensor.view(-1, group_size)
print(test_tensor)

tensor([[0.7569, 0.5203, 0.4888, 0.3090],
        [0.6120, 0.2026, 0.3202, 0.9086],
        [0.1879, 0.5149, 0.0355, 0.8984],
        [0.0924, 0.3639, 0.9540, 0.0336]])
tensor([[0.7569, 0.5203],
        [0.4888, 0.3090],
        [0.6120, 0.2026],
        [0.3202, 0.9086],
        [0.1879, 0.5149],
        [0.0355, 0.8984],
        [0.0924, 0.3639],
        [0.9540, 0.0336]])
